# Lectura: Document Loaders interesantes en LangChain

Los Document Loaders son uno de los componentes más versátiles y útiles del ecosistema LangChain. Permiten convertir prácticamente cualquier fuente de información en documentos estructurados que pueden ser procesados por modelos de lenguaje. En este artículo exploraremos los loaders más interesantes y útiles, con ejemplos prácticos para cada uno.

---

## ¿Qué son los Document Loaders?

Los Document Loaders son clases especializadas que:

- Extraen contenido de diferentes fuentes (web, archivos, APIs, etc.)
- Convierten el contenido en objetos `Document` estandarizados
- Preservan metadatos importantes sobre la fuente original
- Facilitan el preprocesamiento y la indexación de datos

Cada documento cargado contiene:

- **`page_content`**: El texto extraído
- **`metadata`**: Información sobre la fuente (URL, fecha, tipo de archivo, etc.)

---

## 1. WebBaseLoader — El Poder de la Web

El `WebBaseLoader` es perfecto para extraer contenido de páginas web de forma sencilla.

```python
from langchain_community.document_loaders import WebBaseLoader

# Ejemplo básico: cargar documentación
loader = WebBaseLoader("https://docs.langchain.com/docs/")
docs = loader.load()

print(f"Título: {docs[0].metadata.get('title', 'Sin título')}")
print(f"URL: {docs[0].metadata['source']}")
print(f"Contenido: {docs[0].page_content[:300]}...")

# Ejemplo avanzado: múltiples URLs con configuración personalizada
urls = [
    "https://python.langchain.com/docs/concepts/",
    "https://python.langchain.com/docs/tutorials/",
    "https://python.langchain.com/docs/how_to/"
]

loader = WebBaseLoader(
    web_paths=urls,
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            "div", {"class": ["main-content", "article-content"]}
        )
    )
)
docs = loader.load()

print(f"Páginas cargadas: {len(docs)}")
for i, doc in enumerate(docs):
    print(f"Página {i+1}: {doc.metadata['source']}")
    print(f"Longitud: {len(doc.page_content)} caracteres")
```

**Casos de uso ideales:**

- Documentación técnica
- Artículos de blog
- Noticias y contenido web estático
- Scraping de contenido público

---

## 2. PyPDFLoader — Documentos PDF Inteligentes

Los PDFs contienen información valiosa que este loader puede extraer página por página.

```python
from langchain_community.document_loaders import PyPDFLoader

# Cargar PDF con información detallada por página
loader = PyPDFLoader("manual_usuario.pdf")
pages = loader.load_and_split()

print(f"Total de páginas: {len(pages)}")

# Analizar contenido por página
for i, page in enumerate(pages[:3]):  # Primeras 3 páginas
    print(f"\n=== PÁGINA {i+1} ===")
    print(f"Número de página: {page.metadata['page']}")
    print(f"Archivo fuente: {page.metadata['source']}")
    print(f"Contenido: {page.page_content[:200]}...")

    # Estadísticas de la página
    words = len(page.page_content.split())
    chars = len(page.page_content)
    print(f"Palabras: {words}, Caracteres: {chars}")

# Buscar páginas con contenido específico
keyword = "configuración"
relevant_pages = [
    page for page in pages
    if keyword.lower() in page.page_content.lower()
]
print(f"\nPáginas que mencionan '{keyword}': {len(relevant_pages)}")
```

**Casos de uso ideales:**

- Manuales técnicos
- Contratos y documentos legales
- Papers académicos
- Reportes empresariales

---

## 3. DirectoryLoader — Procesamiento Masivo

Para procesar múltiples archivos de forma eficiente.

```python
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_community.document_loaders import UnstructuredMarkdownLoader
import os

# Cargar todos los archivos markdown de un proyecto
loader = DirectoryLoader(
    'docs/',
    glob="**/*.md",
    loader_cls=UnstructuredMarkdownLoader,
    recursive=True,
    show_progress=True,
    use_multithreading=True
)

docs = loader.load()
print(f"Documentos cargados: {len(docs)}")

# Análisis del contenido cargado
total_chars = sum(len(doc.page_content) for doc in docs)
file_stats = {}

for doc in docs:
    filename = os.path.basename(doc.metadata['source'])
    file_stats[filename] = {
        'chars': len(doc.page_content),
        'words': len(doc.page_content.split()),
        'lines': doc.page_content.count('\n') + 1
    }

# Mostrar estadísticas
print(f"\nTotal de caracteres procesados: {total_chars:,}")
print("\nTop 5 archivos más largos:")
sorted_files = sorted(file_stats.items(), key=lambda x: x[1]['chars'], reverse=True)
for filename, stats in sorted_files[:5]:
    print(f"  {filename}: {stats['chars']:,} chars, {stats['words']:,} words")
```

**Casos de uso ideales:**

- Bases de conocimiento empresariales
- Repositorios de código con documentación
- Colecciones de artículos
- Archivos de proyectos completos

---

## 4. YoutubeLoader — Contenido Multimedia

Extrae transcripciones automáticas de videos de YouTube.

```python
from langchain_community.document_loaders import YoutubeLoader
import re

# Cargar transcripción de un video educativo
video_url = "https://www.youtube.com/watch?v=ejemplo123"
loader = YoutubeLoader.from_youtube_url(
    video_url,
    add_video_info=True,
    language=['es', 'en'],  # Priorizar idiomas
    translation='es'        # Traducir si es necesario
)

try:
    docs = loader.load()
    video_info = docs[0].metadata
    transcript = docs[0].page_content

    print("=== INFORMACIÓN DEL VIDEO ===")
    print(f"Título: {video_info.get('title', 'N/A')}")
    print(f"Autor: {video_info.get('author', 'N/A')}")
    print(f"Duración: {video_info.get('length', 'N/A')} segundos")
    print(f"Fecha de publicación: {video_info.get('publish_date', 'N/A')}")
    print(f"Vistas: {video_info.get('view_count', 'N/A')}")

    print(f"\n=== ANÁLISIS DE TRANSCRIPCIÓN ===")
    print(f"Longitud de transcripción: {len(transcript):,} caracteres")
    print(f"Palabras aproximadas: {len(transcript.split()):,}")

    # Extraer temas principales (palabras frecuentes)
    words = re.findall(r'\b[a-záéíóúñ]{4,}\b', transcript.lower())
    from collections import Counter
    common_words = Counter(words).most_common(10)

    print("\nPalabras más frecuentes:")
    for word, count in common_words:
        print(f"  {word}: {count} veces")

    print(f"\nPrimeros 500 caracteres:")
    print(transcript[:500] + "...")

except Exception as e:
    print(f"Error al cargar el video: {e}")
```

**Casos de uso ideales:**

- Análisis de contenido educativo
- Transcripción de conferencias
- Análisis de tendencias en videos
- Creación de resúmenes automáticos

---

## 5. UnstructuredHTMLLoader — HTML Avanzado

Procesa archivos HTML locales o remotos con análisis inteligente de estructura.

```python
from langchain_community.document_loaders import UnstructuredHTMLLoader

# Procesar archivo HTML local
loader = UnstructuredHTMLLoader(
    "reporte_anual.html",
    mode="elements",  # Preservar estructura de elementos
    strategy="fast"   # Estrategia de procesamiento
)

docs = loader.load()

print(f"Elementos HTML procesados: {len(docs)}")

# Analizar tipos de elementos encontrados
element_types = {}
for doc in docs:
    element_type = doc.metadata.get('category', 'unknown')
    element_types[element_type] = element_types.get(element_type, 0) + 1

print("\nTipos de elementos encontrados:")
for element_type, count in sorted(element_types.items()):
    print(f"  {element_type}: {count}")

# Mostrar elementos de texto más largos
text_elements = [doc for doc in docs if doc.metadata.get('category') == 'NarrativeText']
text_elements.sort(key=lambda x: len(x.page_content), reverse=True)

print(f"\nTop 3 elementos de texto más largos:")
for i, element in enumerate(text_elements[:3]):
    print(f"{i+1}. {len(element.page_content)} caracteres:")
    print(f"   {element.page_content[:150]}...")
```

**Casos de uso ideales:**

- Procesamiento de reportes web
- Análisis de documentación HTML
- Extracción de contenido de emails HTML
- Procesamiento de archivos exportados

---

## 6. CSVLoader — Datos Tabulares

Convierte datos estructurados en documentos procesables.

```python
from langchain_community.document_loaders import CSVLoader

# Configuración avanzada para CSV
loader = CSVLoader(
    file_path="ventas_2024.csv",
    csv_args={
        'delimiter': ',',
        'quotechar': '"',
        'fieldnames': ['fecha', 'producto', 'cantidad', 'precio', 'cliente']
    },
    encoding='utf-8',
    source_column='producto',  # Usar una columna como identificador
    metadata_columns=['fecha', 'cliente']  # Incluir en metadatos
)

docs = loader.load()
print(f"Registros de ventas cargados: {len(docs)}")

# Análisis de los datos cargados
productos = set()
clientes = set()
ventas_por_fecha = {}

for doc in docs[:10]:  # Mostrar primeros 10 registros
    print(f"\nRegistro: {doc.page_content}")
    print(f"Metadatos: {doc.metadata}")

    # Recopilar estadísticas
    if 'fecha' in doc.metadata:
        fecha = doc.metadata['fecha']
        ventas_por_fecha[fecha] = ventas_por_fecha.get(fecha, 0) + 1

    if 'cliente' in doc.metadata:
        clientes.add(doc.metadata['cliente'])

print(f"\nResumen de datos:")
print(f"  Clientes únicos: {len(clientes)}")
print(f"  Fechas con ventas: {len(ventas_por_fecha)}")
print(f"  Promedio de ventas por día: {len(docs) / len(ventas_por_fecha):.1f}")
```

**Casos de uso ideales:**

- Análisis de datos de ventas
- Logs de sistema
- Datos de encuestas
- Registros de transacciones

---

## 7. SeleniumURLLoader — JavaScript y Contenido Dinámico

Para páginas web que requieren renderizado de JavaScript.

```python
from langchain_community.document_loaders import SeleniumURLLoader
from selenium.webdriver.chrome.options import Options

# Configurar opciones de navegador
chrome_options = Options()
chrome_options.add_argument("--headless")  # Sin interfaz gráfica
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")

# URLs con contenido dinámico
urls = [
    "https://example.com/dashboard-dinamico",
    "https://spa-application.com/data",
    "https://chart-website.com/interactive"
]

loader = SeleniumURLLoader(
    urls=urls,
    browser="chrome",
    executable_path="/path/to/chromedriver",  # Opcional si está en PATH
    chrome_options=chrome_options
)

docs = loader.load()

print(f"Páginas procesadas: {len(docs)}")

for i, doc in enumerate(docs):
    print(f"\n=== PÁGINA {i+1} ===")
    print(f"URL: {doc.metadata['source']}")
    print(f"Título: {doc.metadata.get('title', 'Sin título')}")
    print(f"Contenido renderizado: {len(doc.page_content)} caracteres")

    # Buscar elementos que indiquen contenido dinámico
    dynamic_indicators = ['data-', 'ng-', 'v-', 'react-', 'vue-']
    has_dynamic = any(indicator in doc.page_content for indicator in dynamic_indicators)
    print(f"Contiene elementos dinámicos: {'Sí' if has_dynamic else 'No'}")

    print(f"Vista previa: {doc.page_content[:200]}...")
```

**Casos de uso ideales:**

- Single Page Applications (SPAs)
- Dashboards interactivos
- Sitios con autenticación
- Contenido generado por JavaScript

---

## 8. GitLoader — Repositorios de Código

Carga archivos directamente desde repositorios Git.

```python
from langchain_community.document_loaders import GitLoader

# Clonar y procesar repositorio
loader = GitLoader(
    clone_url="https://github.com/usuario/proyecto-ejemplo.git",
    repo_path="./temp_repo",
    branch="main",
    file_filter=lambda file_path: file_path.endswith(('.py', '.md', '.txt'))
)

docs = loader.load()
print(f"Archivos cargados del repositorio: {len(docs)}")

# Análisis por tipo de archivo
file_types = {}
total_lines = 0

for doc in docs:
    file_path = doc.metadata['source']
    file_ext = file_path.split('.')[-1] if '.' in file_path else 'sin_extension'

    file_types[file_ext] = file_types.get(file_ext, 0) + 1
    lines = doc.page_content.count('\n') + 1
    total_lines += lines

    print(f"Archivo: {file_path}")
    print(f"  Líneas: {lines}")
    print(f"  Caracteres: {len(doc.page_content)}")
    print(f"  Vista previa: {doc.page_content[:100]}...")
    print()

print(f"\nEstadísticas del repositorio:")
print(f"  Total de líneas de código: {total_lines:,}")
print(f"  Tipos de archivo:")
for ext, count in sorted(file_types.items()):
    print(f"    .{ext}: {count} archivos")
```

**Casos de uso ideales:**

- Análisis de código fuente
- Documentación de proyectos
- Auditorías de repositorios
- Generación de documentación automática


# Text splitters

#### Este código supera la ventana de contexto del LLM

In [ ]:
import httpx
from dotenv import load_dotenv,find_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_openai import ChatOpenAI

In [ ]:
load_dotenv(find_dotenv())
http_client = httpx.Client(verify=False)

In [ ]:
# 1. Cargar el documento PDF
loader = PyPDFLoader("quijote.pdf")
pages = loader.load()

In [ ]:
# 2. Combinar todas las paginas en un texto unico
full_text = ""
for page in pages:
    full_text += page.page_content + "\n"

In [ ]:
# 3. Pasar el texto al LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.2,
    http_client=http_client
)

In [ ]:
# Esta peticion geneará un error porque el texto es demasiado largo para ser procesado por el modelo.
response = llm.invoke(f"Haz un resumen de los puntos mas importantes del siguiente documento: {full_text}")

print(response.content)

#### Solución al problema de ventana de contexto

In [9]:
import httpx

from dotenv import load_dotenv,find_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_openai import ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter # Solucionar el problema de texto largo dividiendolo en partes mas pequeñas

In [10]:
load_dotenv(find_dotenv())
http_client = httpx.Client(verify=False)

In [12]:
# 1. Cargar el documento PDF
loader = PyPDFLoader("recursos/quijote.pdf")
pages = loader.load()

In [13]:
# Dividir el texto en chunks mas pequeños para evitar el error de texto largo
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000, # Tamaño máximo de cada chunk expresado en caracteres
    chunk_overlap=200 # Superposición entre chunks
)

In [15]:
# Dividir cada pagina en chunks mas pequeños con la configuracion anterior
chunks = text_splitter.split_documents(pages)
print(len(chunks)) # Imprimir la cantidad de chunks generados

734


In [16]:
# 3. Pasar el texto al LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.2,
    http_client=http_client
)

In [8]:
summaries = []
for chunk in chunks:
    response = llm.invoke(f"Haz un resumen de los puntos mas importantes del siguiente texto: {chunk.page_content}")
    summaries.append(response.content)

KeyboardInterrupt: 

In [ ]:
final_summary = llm.invoke(f"Combina y sintetiza estos resumenes en un resumen coherente y completo: {' '.join(summaries)}")

In [ ]:
print(final_summary.content)

# Embeddings

```text
Un embedding es una representación numérica de un texto que captura su significado semántico. 
Los modelos de embeddings transforman palabras, frases o documentos en vectores de alta dimensión, 
donde la proximidad entre los vectores refleja la similitud semántica entre los textos originales. 
Esto permite a las máquinas entender y comparar el significado de los textos, facilitando tareas como 
la búsqueda, clasificación y generación de lenguaje natural.
```

In [17]:
import httpx

from dotenv import load_dotenv,find_dotenv
import numpy as np

from langchain_openai import OpenAIEmbeddings

In [18]:
load_dotenv(find_dotenv())
http_client = httpx.Client(verify=False)

In [19]:
# Crear una instancia de OpenAIEmbeddings
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-large", http_client=http_client
)

In [20]:
texto1 = "La capital de Francia es París."
texto2 = "París es la ciuidad capital de Francia."

In [21]:
# Obtener los vectores de embeddings para ambos textos
vector1 = embeddings.embed_query(texto1)
vector2 = embeddings.embed_query(texto2)

In [22]:
print(f"Dimensiones del vector 1: {len(vector1)}")
print(f"Dimensiones del vector 2: {len(vector2)}")

Dimensiones del vector 1: 3072
Dimensiones del vector 2: 3072


Calcular la similitud coseno entre los dos vectores a través de la fórmula:

$$\text{cos\_sim} = \frac{A \cdot B}{\|A\| \times \|B\|}$$

Donde:

- $A$ y $B$ son los vectores de *embeddings*
- $\cdot$ representa el producto punto
- $\|A\|$ y $\|B\|$ son las normas (magnitudes) de los vectores

También se puede usar la función de similitud coseno de `sklearn.metrics.pairwise.cosine_similarity`, pero aquí lo calculamos manualmente para entender el proceso.

In [23]:
cos_sim = np.dot(vector1, vector2) / (np.linalg.norm(vector1) * np.linalg.norm(vector2))

In [24]:
print(f"Similitud coseno entre los dos textos: {cos_sim:.3f}")

Similitud coseno entre los dos textos: 0.836


# Vector stores

```text
Los vector stores son estructuras de datos que almacenan vectores de embeddings junto con sus metadatos asociados. 
Permiten realizar búsquedas eficientes y recuperar información relevante basada en la similitud semántica entre 
los vectores.
```

In [ ]:
# En este ejemplo, se utiliza Chroma como vector store para almacenar los embeddings generados a partir 
# de documentos PDF, lo que facilita la búsqueda de información específica dentro de esos documentos.

In [39]:
import httpx
import os

from dotenv import load_dotenv,find_dotenv

from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [40]:
load_dotenv(find_dotenv())
http_client = httpx.Client(verify=False)

In [41]:
loader = PyPDFDirectoryLoader("recursos\\contratos")
documentos = loader.load()

In [43]:
print(f"se cargaron {len(documentos)} documentos desde el directorio")

se cargaron 15 documentos desde el directorio


In [31]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=5000, 
    chunk_overlap=1000
)

In [32]:
docs_split = text_splitter.split_documents(documentos)

In [33]:
print(f"se generaron {len(docs_split)} fragmentos de texto después de la división")

se generaron 15 fragmentos de texto después de la división


In [34]:
vector_store = Chroma.from_documents(
    documents=docs_split,
    embedding=OpenAIEmbeddings(model="text-embedding-3-large", http_client=http_client),
    persist_directory="chroma_database"
)

In [35]:
consulta = "¿Donde se enceuntra el local del contrato en el que participa María Jiménez Campos?"

In [36]:
resultados = vector_store.similarity_search(consulta, k=2)

In [38]:
print("Top 2 documentos mas similares a la consulta:")
for i, doc in enumerate(resultados, 1):
    print(f"Contenido del documento {i}:\n{doc.page_content}\n")
    print(f"\n\n\n")
    print(f"Metadatos del documento {i}:\n{doc.metadata}\n")

Top 2 documentos mas similares a la consulta:
Contenido del documento 1:
CONTRATO DE ARRENDAMIENTO DE LOCAL DE NEGOCIO 
En Sevilla, a 28 de mayo de 2025 
REUNIDOS 
De una parte, Dª. María Jiménez Campos, mayor de edad, con DNI 45 678 901-G, con domicilio 
a efectos de notificaciones en Avenida de la Constitución 12, 1.º A, 41004 Sevilla (España), en 
adelante LA ARRENDADORA. 
Y de otra parte, Tienda Verde, S.L., con CIF B12345678, domiciliada en Calle San Jacinto 58, 
41010 Sevilla y representada en este acto por su administrador único D. Pablo Ruiz Calvo, 
mayor de edad, con NIF 12 345 678-H, en adelante LA ARRENDATARIA. 
Ambas partes, reconociéndose capacidad legal suficiente, MANIFIESTAN: 
1. Que LA ARRENDADORA es propietaria en pleno dominio del local comercial nº 3 
situado en la planta baja del edificio en Calle Tetuán 15, 41001 Sevilla, inscrito en el 
Registro de la Propiedad de Sevilla nº 6, tomo 9 876, libro 1 234, folio 210, 
finca 65 432. 
2. Que LA ARRENDATARIA está intere

# Retrivers

```text
Los retrievers son componentes que permiten recuperar información relevante de un vector store "basado en una consulta". 
```

In [ ]:
# En este ejemplo, se utiliza el método as_retriever de Chroma para crear un retriever que realiza búsquedas de similitud 
# en el vector store, devolviendo los documentos más relevantes para una consulta dada. Esto facilita la recuperación de 
# información específica dentro de un conjunto de documentos almacenados

In [ ]:
vector_store = Chroma(
    embedding_function=OpenAIEmbeddings(model="text-embedding-3-large", http_client=http_client),
    persist_directory="chroma_database/"
)

In [ ]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 2})

In [ ]:
consulta = "¿Donde se enceuntra el local del contrato en el que participa María Jiménez Campos?"

In [ ]:
resultados = retriever.invoke(consulta)

In [ ]:
print("Top 2 documentos mas similares a la consulta: \n")
for i, doc in enumerate(resultados, start=1):
    print(f"Contenido del documento {i}:\n{doc.page_content}\n")
    print(f"Metadatos del documento {i}:\n{doc.metadata}\n")

## Multiquery retriever

```text
El multi-query retriever es una extensión de los retrievers tradicionales que permite realizar múltiples 
consultas para recuperar información más relevante y diversa
```

In [ ]:
# En este ejemplo, se utiliza un modelo de lenguaje para generar consultas adicionales basadas en la consulta
# original del usuario, lo que mejora la capacidad de recuperación de información al considerar diferentes 
# formulaciones y perspectivas de la misma pregunta.

In [ ]:
import httpx

from dotenv import load_dotenv,find_dotenv

from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings,ChatOpenAI
from langchain.retrievers.multi_query import MultiQueryRetriever

In [ ]:
vector_store = Chroma(
    embedding_function=OpenAIEmbeddings(model="text-embedding-3-large", http_client=http_client),
    persist_directory="chroma_database/"
)

In [ ]:
llm = ChatOpenAI(
    model="gpt-4o-mini", 
    temperature=0,
    http_client=http_client
)

In [ ]:
base_retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 2})
retriever = MultiQueryRetriever.from_llm(retriever=base_retriever, llm=llm)

In [ ]:
consulta = "¿Donde se encuentra el local del contrato en el que participa María Jiménez Campos?"

In [ ]:
resultados = retriever.invoke(consulta)

print("Top documentos mas similares a la consulta: \n")
for i, doc in enumerate(resultados, start=1):
    print(f"Contenido del documento {i}:\n{doc.page_content}\n")
    print(f"Metadatos del documento {i}:\n{doc.metadata}\n")

## Retrievers Interesantes en LangChain

Los retrievers constituyen el corazón de cualquier sistema RAG (Retrieval-Augmented Generation) exitoso. LangChain ha evolucionado para ofrecer una amplia gama de retrievers sofisticados que van mucho más allá de la simple búsqueda por similitud. En este artículo, exploraremos los retrievers más interesantes y potentes que están transformando la manera en que nuestras aplicaciones de IA acceden y procesan información.

---

### ¿Qué es un Retriever en LangChain?

Un retriever en LangChain es una interfaz que recibe una consulta en lenguaje natural y devuelve documentos relevantes. A diferencia de los vector stores, los retrievers no necesitan almacenar documentos, solo recuperarlos. Esta abstracción permite crear sistemas de recuperación flexibles y modulares que se adaptan a diferentes necesidades.

**Características clave:**

- Entrada: string de consulta
- Salida: lista de objetos `Document`
- Implementan la interfaz `Runnable` estándar
- Compatibles con LCEL (LangChain Expression Language)

---

### 1. MultiQueryRetriever: La Perspectiva Múltiple

#### ¿Qué hace?

El `MultiQueryRetriever` aborda las limitaciones de la búsqueda por similitud basada en distancia generando múltiples "perspectivas" alternativas de tu consulta original. En lugar de hacer una sola búsqueda, utiliza un LLM para crear varias versiones de la misma pregunta y luego combina los resultados.

#### Implementación

```python
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_openai import ChatOpenAI

# Configurar el retriever
llm = ChatOpenAI(temperature=0)
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=vector_store.as_retriever(),
    llm=llm,
    verbose=True  # Ver las consultas generadas
)

# Una pregunta se convierte en múltiples perspectivas
results = multi_query_retriever.invoke("¿Cómo aprenden las redes neuronales?")
```

#### Cuándo usarlo

- Cuando tu consulta original puede ser interpretada de múltiples formas
- Para mejorar la diversidad de resultados recuperados
- En casos donde la consulta inicial podría no capturar todos los aspectos relevantes

---

### 2. ContextualCompressionRetriever: El Filtro Inteligente

#### ¿Qué problema resuelve?

Uno de los desafíos con la recuperación es que normalmente no conoces las consultas específicas que enfrentará tu sistema de almacenamiento de documentos cuando ingieres datos al sistema. Esto significa que la información más relevante para una consulta puede estar enterrada en un documento con mucho texto irrelevante.

#### Implementación

```python
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

# Crear compresor que extrae solo contenido relevante
compressor = LLMChainExtractor.from_llm(llm)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=vector_store.as_retriever()
)

# Solo obtener las partes relevantes
compressed_results = compression_retriever.invoke(
    "Explica el algoritmo de retropropagación"
)
```

#### Pipeline avanzado de compresión

```python
from langchain.retrievers.document_compressors import DocumentCompressorPipeline
from langchain_community.document_transformers import EmbeddingsRedundantFilter
from langchain_text_splitters import CharacterTextSplitter

# Pipeline: dividir -> filtrar redundantes -> comprimir por relevancia
splitter = CharacterTextSplitter(chunk_size=300, chunk_overlap=0, separator=".")
redundant_filter = EmbeddingsRedundantFilter(embeddings=embeddings)
relevant_filter = LLMChainExtractor.from_llm(llm)

pipeline_compressor = DocumentCompressorPipeline(
    transformers=[splitter, redundant_filter, relevant_filter]
)
```

#### Beneficios

- Reduce costos de llamadas a LLM al eliminar texto irrelevante
- Mejora la calidad de las respuestas al proporcionar contexto más preciso
- Permite pasar más documentos relevantes dentro del límite de tokens

---

### 3. EnsembleRetriever: El Mejor de Dos Mundos

#### La potencia del híbrido

El `EnsembleRetriever` soporta la combinación de resultados de múltiples retrievers. Los EnsembleRetrievers reordenan los resultados de los retrievers constituyentes basándose en el algoritmo de Fusión de Ranking Recíproco.

#### Implementación BM25 + Vector Search

```python
from langchain.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

# Crear retriever BM25 (búsqueda por palabras clave)
bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = 5

# Crear retriever vectorial (búsqueda semántica)
embedding = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(documents, embedding)
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# Combinar ambos con pesos
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.3, 0.7]  # 30% BM25, 70% vectorial
)
```

#### Por qué es efectivo

- **BM25:** Excelente para coincidencias exactas de palabras clave
- **Vector Search:** Superior para similitud semántica
- **Fusión:** Combina las fortalezas de ambos enfoques

#### Casos de uso ideales

- Búsquedas que requieren tanto precisión léxica como semántica
- Documentos técnicos con terminología específica
- Sistemas de búsqueda empresarial

---

### 4. ParentDocumentRetriever: Precisión con Contexto

#### El equilibrio perfecto

El `ParentDocumentRetriever` logra ese equilibrio dividiendo y almacenando pequeños trozos de datos. Durante la recuperación, primero obtiene los trozos pequeños pero luego busca los IDs padre de esos trozos y devuelve esos documentos más grandes.

#### Implementación

```python
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Splitter para documentos padre (chunks grandes)
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000)

# Splitter para documentos hijo (chunks pequeños para embedding)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400)

# Vector store para los chunks pequeños
vectorstore = Chroma(
    collection_name="parent_docs",
    embedding_function=OpenAIEmbeddings()
)

# Almacenamiento para documentos padre
store = InMemoryStore()

retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

# Agregar documentos
retriever.add_documents(documents)
```

#### Ventajas clave

- **Embeddings precisos:** Los chunks pequeños crean embeddings más representativos
- **Contexto completo:** Devuelve documentos padre con contexto suficiente
- **Flexibilidad:** Puedes ajustar el tamaño de chunks padre e hijo independientemente

---

### 5. SelfQueryRetriever: Búsqueda Estructurada Inteligente

#### Más allá de la similitud semántica

`SelfQueryRetriever` utilizará un LLM para generar una consulta que es potencialmente estructurada, por ejemplo, puede construir filtros para la recuperación además de la selección habitual dirigida por similitud semántica.

#### Implementación

```python
from langchain.chains.query_constructor.base import AttributeInfo
from langchain.retrievers.self_query.base import SelfQueryRetriever

# Definir metadatos de los documentos
metadata_field_info = [
    AttributeInfo(
        name="genre",
        description="El género de la película",
        type="string"
    ),
    AttributeInfo(
        name="year",
        description="El año de lanzamiento de la película",
        type="integer"
    ),
    AttributeInfo(
        name="rating",
        description="La calificación de la película (1-10)",
        type="float"
    ),
]

document_content_description = "Breve resumen de una película"

retriever = SelfQueryRetriever.from_llm(
    llm=ChatOpenAI(temperature=0),
    vectorstore=vectorstore,
    document_content_description=document_content_description,
    metadata_field_info=metadata_field_info,
)

# Consulta que se convertirá en filtros estructurados
results = retriever.invoke("películas de ciencia ficción de después de 2010 con calificación alta")
```

#### Casos de uso

- Bases de datos con metadatos ricos
- Consultas que combinan contenido y filtros
- Sistemas que requieren búsqueda estructurada automática

---

### 6. TimeWeightedVectorStoreRetriever: Memoria que Desvanece

#### Para información sensible al tiempo

Este retriever asigna mayor importancia a documentos más recientes, simulando cómo funciona la memoria humana.

```python
from langchain.retrievers import TimeWeightedVectorStoreRetriever
from datetime import datetime, timedelta

tw_retriever = TimeWeightedVectorStoreRetriever(
    vectorstore=vectorstore,
    decay_rate=0.999,  # Qué tan rápido "olvida" documentos antiguos
    k=5
)

# Los documentos más antiguos tendrán menos peso
yesterday = datetime.now() - timedelta(days=1)
tw_retriever.add_documents([Document(page_content="Noticia vieja")], timestamps=[yesterday])
tw_retriever.add_documents([Document(page_content="Noticia nueva")])
```

---

### 7. Técnicas Avanzadas y Combinaciones

#### Retrieval con Reranking

```python
from langchain.retrievers.document_compressors import CohereRerank

# Primer pase: recuperar muchos documentos
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 20})

# Segundo pase: reordenar con Cohere
reranker = CohereRerank(
    model="rerank-multilingual-v2.0",
    top_n=5
)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=base_retriever
)
```

#### Retrieval MMR (Maximum Marginal Relevance)

```python
# Balancear relevancia con diversidad
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 10,
        "lambda_mult": 0.7  # Balance entre relevancia (1.0) y diversidad (0.0)
    }
)
```

---

### Conclusión

Los retrievers de LangChain han evolucionado significativamente, ofreciendo soluciones especializadas para cada desafío de recuperación de información. La clave del éxito está en:

1. **Entender tu caso de uso:** Cada retriever tiene fortalezas específicas
2. **Experimentar con combinaciones:** Los mejores sistemas suelen combinar múltiples técnicas
3. **Evaluar constantemente:** La calidad de recuperación impacta directamente en la experiencia del usuario
4. **Optimizar para producción:** Considera latencia, costos y escalabilidad

El ecosistema de retrievers de LangChain seguirá evolucionando, pero dominar estos conceptos fundamentales te dará una base sólida para construir aplicaciones RAG verdaderamente efectivas.

> ¿Qué retriever vas a implementar primero en tu próximo proyecto? La elección correcta puede marcar la diferencia entre una aplicación que funciona y una que realmente sobresale.
